# Keeks strategy lab

Swap bet-sizing rules under identical conditions and compare what each one does to a bankroll. Nothing to install locally: run the cells top to bottom.

> **Educational use only.** This notebook simulates a made-up repeated bet with a probability you set. It is not betting, investment, legal or tax advice, and its output is not a forecast of any real result. Keeks is an alpha-stage educational library.

What you will see: four sizing rules face the same sequence of outcomes on every path, and the notebook charts the *distribution* of terminal balance and of maximum drawdown. There is no winner. Growth and drawdown pull in different directions, and the point is to see the trade.

In [ ]:
%pip install -q "keeks==0.8.0"

## 1. Parameters

Everything you may want to change is in this one cell. Two of the names look alike and are **not** the same thing:

| Name | Where it goes | Unit |
|---|---|---|
| `STRATEGY_TRANSACTION_COST` | `transaction_cost_rate` (singular) on each strategy | fraction of each unit staked, used only for sizing |
| `SIMULATOR_TRANSACTION_COSTS` | `fee_per_bet` (plural) on `RepeatedBinarySimulator` | absolute bankroll amount charged once per settled bet |

The strategy's number changes how much it stakes. The simulator's number changes what a settled bet costs. Passing the same value to both models two different costs. This lab sets them independently so you can see each effect.

In [ ]:
PROBABILITY = 0.55  # chance each bet wins (an assumption you choose, not an estimate of anything real)
PAYOFF = 1.0  # win 1.0 per unit staked
LOSS = 1.0  # lose 1.0 per unit staked
STRATEGY_TRANSACTION_COST = 0.01  # singular, fractional: enters the sizing formulas
SIMULATOR_TRANSACTION_COSTS = (
    0.25  # plural, absolute amount per settled bet: charged by the simulator
)

INITIAL_BANKROLL = 1000.0
N_BETS = 200  # bets per path
N_PATHS = 300  # paired paths
MASTER_SEED = 2024

FRACTIONAL_KELLY = 0.5
FIXED_FRACTION = 0.05

## 2. Strategies and fresh state

Strategies are built by a factory, and **every path gets a new strategy and a new `BankRoll`**. Two things carry state:

- `BankRoll` accumulates `history` and enforces its drawdown limit as it goes.
- `DynamicBankrollManagement` (the drawdown-aware strategy here) keeps a window of recent results via `record_settlement`, so its stake on bet *n* depends on bets before it.

Reusing either object across paths silently leaks one path into the next. Section 6 demonstrates the effect.

`BankRoll(max_transaction_loss=None)` is set here so no path is cut short. A `0.3` setting caps any *single* withdrawal relative to current funds, not a cumulative drawdown budget; a breach stops the path early (the `None` set here removes the cap).

In [ ]:
import itertools
import random

import matplotlib.pyplot as plt
import numpy as np

import keeks
from keeks.bankroll import BankRoll
from keeks.binary_strategies.kelly import FractionalKellyCriterion, KellyCriterion
from keeks.binary_strategies.simple import (
    DynamicBankrollManagement,
    FixedFractionStrategy,
)
from keeks.simulators.repeated_binary import RepeatedBinarySimulator

print(
    "keeks", keeks.__version__ if hasattr(keeks, "__version__") else "(no __version__)"
)


def make_strategies():
    kw = {
        "payoff": PAYOFF,
        "loss": LOSS,
        "transaction_cost_rate": STRATEGY_TRANSACTION_COST,
    }
    return {
        "Kelly": KellyCriterion(**kw),
        f"{FRACTIONAL_KELLY:g}x Kelly": FractionalKellyCriterion(
            fraction=FRACTIONAL_KELLY, **kw
        ),
        f"Fixed {FIXED_FRACTION:.0%}": FixedFractionStrategy(
            fraction=FIXED_FRACTION, **kw
        ),
        "Dynamic (drawdown-aware)": DynamicBankrollManagement(
            base_fraction=FIXED_FRACTION, min_fraction=0.01, max_fraction=0.10, **kw
        ),
    }


NAMES = list(make_strategies())
print("Stake fraction at the assumed probability, before any bankroll state:")
for name, s in make_strategies().items():
    print(f"  {name:26s} {s.evaluate(PROBABILITY, INITIAL_BANKROLL):.4f}")

## 3. Paired paths

`RepeatedBinarySimulator` draws an outcome **only when the strategy stakes something**. If one strategy skips a bet and another does not, a single seeded simulator would hand them different outcomes from that point on, and the comparison would no longer be like for like.

To keep paths paired, each bet gets its own seed derived from `(MASTER_SEED, path, bet index)`, and the simulator runs one bet at a time. Bet *t* of path *i* then has the same outcome for every strategy whether or not they staked earlier bets. The same `BankRoll` and strategy objects carry through the whole path.

In [ ]:
def bet_seed(path, t):
    return int(np.random.SeedSequence([MASTER_SEED, path, t]).generate_state(1)[0])


def run_path(strategy, path, fee=None, max_transaction_loss=None):
    fee = SIMULATOR_TRANSACTION_COSTS if fee is None else fee
    bankroll = BankRoll(INITIAL_BANKROLL, max_transaction_loss=max_transaction_loss)
    for t in range(N_BETS):
        sim = RepeatedBinarySimulator(
            payoff=PAYOFF,
            loss=LOSS,
            fee_per_bet=fee,
            probability=PROBABILITY,
            trials=1,
            seed=bet_seed(path, t),
        )
        sim.evaluate_strategy(strategy, bankroll)
    return bankroll


def max_drawdown(history):
    h = np.asarray(history, dtype=float)
    peak = np.maximum.accumulate(h)
    return float(np.max(np.where(peak > 0, (peak - h) / peak, 0.0)))


def run_all(n_paths=None):
    n_paths = N_PATHS if n_paths is None else n_paths
    terminal = {n: [] for n in NAMES}
    drawdown = {n: [] for n in NAMES}
    for path in range(n_paths):
        for name, strategy in make_strategies().items():  # fresh strategy per path
            bankroll = run_path(strategy, path)  # fresh BankRoll per path
            terminal[name].append(bankroll.total_funds)
            drawdown[name].append(max_drawdown(bankroll.history))
    return {n: np.array(v) for n, v in terminal.items()}, {
        n: np.array(v) for n, v in drawdown.items()
    }


terminal, drawdown = run_all()
print(f"{N_PATHS} paths x {N_BETS} bets x {len(NAMES)} strategies done")

**Pairing check, including a strategy that skips bets.** `SkipEveryOther` below stakes only on every second bet. With the fee at zero, each settled bet moves the bankroll up on a win and down on a loss, so the direction of every move can be compared with the shared outcome of that bet index. The skipper's moves must equal the shared outcomes at the bets it staked, and a strategy that stakes every bet must match all of them. The last line shows what a single seeded simulator would have done instead: the skipper's second stake receives the *first* draw, not the second.

In [ ]:
class SkipEveryOther(FixedFractionStrategy):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.calls = 0

    def evaluate(self, probability, current_bankroll):
        self.calls += 1
        return (
            super().evaluate(probability, current_bankroll)
            if self.calls % 2 == 0
            else 0.0
        )


def shared_outcome(path, t):
    return random.Random(bet_seed(path, t)).random() < PROBABILITY


def moves_of(strategy, path):
    history = run_path(strategy, path, fee=0.0).history
    return [b > a for a, b in itertools.pairwise(history)]


for path in range(5):
    wins = [shared_outcome(path, t) for t in range(N_BETS)]
    assert (
        moves_of(KellyCriterion(PAYOFF, LOSS, STRATEGY_TRANSACTION_COST), path) == wins
    )
    skipper = SkipEveryOther(fraction=FIXED_FRACTION, payoff=PAYOFF, loss=LOSS)
    assert moves_of(skipper, path) == wins[1::2]
print(
    "pairing check passed: staked-every-bet and skip-every-other both see the shared outcomes"
)


def single_seed_moves(strategy):
    sim = RepeatedBinarySimulator(PAYOFF, LOSS, 0.0, PROBABILITY, trials=N_BETS, seed=7)
    bank = BankRoll(INITIAL_BANKROLL, max_transaction_loss=None)
    sim.evaluate_strategy(strategy, bank)
    return [b > a for a, b in zip(bank.history, bank.history[1:], strict=False)]


every = single_seed_moves(KellyCriterion(PAYOFF, LOSS, STRATEGY_TRANSACTION_COST))
skip = single_seed_moves(
    SkipEveryOther(fraction=FIXED_FRACTION, payoff=PAYOFF, loss=LOSS)
)
print(
    "one seeded simulator: skipper's outcomes == outcomes of the SAME bet indexes the other strategy staked?",
    skip == every[1::2][: len(skip)],
)
print(
    "one seeded simulator: skipper's outcomes == the other strategy's FIRST draws (stream shifted)?",
    skip == every[: len(skip)],
)

## 4. Terminal balance and drawdown distributions

Each box shows the spread across the same paired paths. Read both panels together: a rule that grows more usually also draws down more. These numbers describe this made-up bet under the parameters above and nothing else.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].boxplot([terminal[n] for n in NAMES], tick_labels=NAMES, showfliers=False)
axes[0].axhline(INITIAL_BANKROLL, color="grey", ls="--", lw=1)
axes[0].set_yscale("log")
axes[0].set_title("Terminal bankroll (log scale; dashed = start)")
axes[1].boxplot([100 * drawdown[n] for n in NAMES], tick_labels=NAMES, showfliers=False)
axes[1].set_title("Maximum drawdown per path (%)")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

print(
    f"{'strategy':26s} {'p10':>9s} {'median':>9s} {'p90':>9s} {'ended<start':>12s} {'median maxDD':>13s} {'p90 maxDD':>10s}"
)
for n in NAMES:
    t, d = terminal[n], drawdown[n]
    print(
        f"{n:26s} {np.percentile(t, 10):9.0f} {np.median(t):9.0f} {np.percentile(t, 90):9.0f} "
        f"{(t < INITIAL_BANKROLL).mean():12.1%} {np.median(d):13.1%} {np.percentile(d, 90):10.1%}"
    )

## 5. Reruns reproduce, other seeds differ

The same `MASTER_SEED` gives the same numbers every time. A different seed gives a different draw of paths, so a single run is one sample, not the answer. Compare the medians and the tails below, and change `MASTER_SEED` to see how far they move.

In [ ]:
N_CHECK = 60
before, _ = run_all(N_CHECK)
again, _ = run_all(N_CHECK)
assert all(np.array_equal(before[n], again[n]) for n in NAMES)
print(f"same seed: identical terminal balances on all {N_CHECK} paths")

MASTER_SEED_SAVED = MASTER_SEED
MASTER_SEED = MASTER_SEED_SAVED + 1
other, _ = run_all(N_CHECK)
MASTER_SEED = MASTER_SEED_SAVED
assert any(not np.array_equal(before[n], other[n]) for n in NAMES)
print(f"different seed, median terminal bankroll over {N_CHECK} paths:")
for n in NAMES:
    print(
        f"  {n:26s} seed {MASTER_SEED_SAVED}: {np.median(before[n]):9.0f}   seed {MASTER_SEED_SAVED + 1}: {np.median(other[n]):9.0f}"
    )

## 6. Gotchas

**Mutation: reusing a stateful strategy.** `DynamicBankrollManagement` remembers recent results. Below, the same instance is run across two paths, and the second path starts with the first path's window. A fresh instance does not.

In [ ]:
kw = {
    "payoff": PAYOFF,
    "loss": LOSS,
    "transaction_cost_rate": STRATEGY_TRANSACTION_COST,
}


def dyn():
    return DynamicBankrollManagement(
        base_fraction=FIXED_FRACTION, min_fraction=0.01, max_fraction=0.10, **kw
    )


reused = dyn()
run_path(reused, 0)  # path 0 leaves its window inside `reused`
leaky = run_path(reused, 1).total_funds  # path 1 inherits it
clean = run_path(dyn(), 1).total_funds  # path 1 with fresh state
print(f"path 1, reused strategy: {leaky:.2f}")
print(f"path 1, fresh strategy:  {clean:.2f}")
assert leaky != clean

**Mutation: reusing a `BankRoll`.** A bankroll carries its funds and `history` forward, so a second path would start from the first path's ending balance.

In [ ]:
bank = BankRoll(INITIAL_BANKROLL, max_transaction_loss=None)
sim = RepeatedBinarySimulator(
    PAYOFF, LOSS, SIMULATOR_TRANSACTION_COSTS, PROBABILITY, trials=N_BETS, seed=1
)
sim.evaluate_strategy(KellyCriterion(**kw), bank)
print("after path A:", bank.total_funds, "history length", len(bank.history))
print(
    "a second path on this object would start at",
    bank.total_funds,
    "not",
    INITIAL_BANKROLL,
)

**Two costs, two units.** The sizing cost shrinks the stake; the simulator fee is charged per settled bet whatever the stake. Measured on the paths above:

In [ ]:
rows = []
for name, strategy in make_strategies().items():
    stake0 = strategy.evaluate(PROBABILITY, INITIAL_BANKROLL) * INITIAL_BANKROLL
    rows.append(
        (
            name,
            stake0,
            100 * SIMULATOR_TRANSACTION_COSTS / stake0 if stake0 else float("nan"),
        )
    )
print(f"{'strategy':26s} {'first stake':>12s} {'sim fee as % of that stake':>28s}")
for r in rows:
    print(f"{r[0]:26s} {r[1]:12.2f} {r[2]:27.2f}%")
print()
print("Raising STRATEGY_TRANSACTION_COST lowers the Kelly-family stakes. Raising")
print(
    "SIMULATOR_TRANSACTION_COSTS does not change any stake; it only changes settlement."
)
print("FixedFractionStrategy ignores STRATEGY_TRANSACTION_COST when sizing.")

**Other limits worth knowing**

- A whole-sample `BankRoll(max_transaction_loss=0.3)` default stops a path on the first single loss above 30% of funds; this lab turns it off.
- Fixed-probability, independent bets only. No correlated positions, no market impact, no execution.
- The probability here is *given to* the strategy and is also the true win rate. In practice an estimate is noisy, and Kelly-style rules react badly to an over-estimated edge.

## 7. Where next

Change the parameter cell and rerun: lower `PROBABILITY` toward `0.5`, add `STRATEGY_TRANSACTION_COST`, or raise `FRACTIONAL_KELLY`. Package docs and source: https://github.com/wdm0006/keeks

*Educational use only. Not betting, investment, legal or tax advice, and not a forecast.*